# UNITPathSSL + DWT-FPN: Full Pipeline (Pretraining → Fine-tuning → Evaluation → Outputs)
Complete Kaggle notebook version of every script in `code/FPN`.

**Pipeline stages:**
1. **Stage A (optional):** Generate synthetic pseudo masks (`generate_pseudo_masks.py`)
2. **Stage B (optional):** UNITPathSSL self-supervised pretraining (`generator_g.py`, `pretrain_large.py`, `loss.py`)
3. **Stage C:** Supervised fine-tuning on MoNuSeg (`dataset.py`, `backbone.py`, `DWTFusion.py`, `ModifiedFPN.py`, `seg_head.py`, `train.py`)
4. **Stage D:** Test evaluation (`test.py`)
5. **Stage E:** Outputs — training curves (`plot_curves.py`), qualitative panels (`generate_output_mask.py`), unlabeled inference (`infer_unlabeled.py`), baseline comparison (`compare_baseline.py`)

**Before running:** Session options → Accelerator = **GPU**, Internet = **On**.

Stages A+B require an unlabeled H&E tissue dataset (e.g., Kather CRC-100K) and ~24h+ of GPU time for the full 65k iterations — they are **skippable**: set `RUN_PRETRAINING = False` to fine-tune directly from ImageNet weights (or from an uploaded UNITPathSSL checkpoint).

In [ ]:
# ============ 1. Imports & environment check ============
import os, glob, time, csv
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, jaccard_score,
                             precision_score, recall_score, roc_auc_score)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============ 2. Configuration (replaces all argparse CLIs) ============
class Config:
    # ---- Pipeline switches ----
    RUN_PRETRAINING = False      # Stage A+B: needs tissue dataset + long GPU time
    RUN_UNLABELED_INFERENCE = False  # Stage E3: needs unlabeled tissue tiles

    # ---- MoNuSeg data (Kaggle paths) ----
    base = "/kaggle/input/datasets/tuanledinh/monuseg2018"
    train_img_dir  = f"{base}/kmms_training/kmms_training/images"
    train_mask_dir = f"{base}/kmms_training/kmms_training/masks"
    test_img_dir   = f"{base}/kmms_test/kmms_test/images"
    test_mask_dir  = f"{base}/kmms_test/kmms_test/masks"

    # ---- Pretraining (Stage A+B) ----
    pseudo_mask_dir = "/kaggle/working/data/pseudo_masks"
    num_pseudo_masks = 5000
    pseudo_mask_size = (256, 256)
    tissue_dir = "/kaggle/input/YOUR_TISSUE_DATASET"  # e.g. Kather CRC-100K tiles
    pretrain_size = (256, 256)
    pretrain_batch_size = 4
    pretrain_accum_steps = 6
    stage1_iters = 40000   # reduce drastically (e.g. 2000) for a Kaggle demo run
    stage2_iters = 25000
    pretrain_save_dir = "/kaggle/working/checkpoints/pretrain"
    # UNITPathSSL objective (paper Eq. 1-3, 7)
    pre_lambda1 = 2.0        # weight of L_GAN(S, D_S)
    pre_lambda2 = 10.0       # weight of L_cyc (both directions)
    pre_lambda3 = 0.0        # weight of L_seg (ISG branch) -- 0 = disabled
    pre_gamma_g = 10.0       # R1 gamma on D_G (real tissue y)
    pre_gamma_s = 10.0       # R1 gamma on D_S (real mask x)
    pre_r1_interval = 16     # lazy R1 every N iters
    use_ada = True           # Adaptive Discriminator Augmentation in front of D_G
    ada_target = 0.6

    # ---- Model ----
    backbone = "resnet50"
    pretrained = True            # ImageNet init for the backbone
    pretrained_weights = None    # path to UNITPathSSL ckpt; auto-filled if RUN_PRETRAINING
    num_classes = 1

    # ---- Fine-tuning (Stage C) — thesis hyperparameters, Kaggle-adapted ----
    lr = 5e-5
    weight_decay = 1e-4
    epochs = 30
    batch_size = 4               # was 1 on 4GB GPU; real batching on Kaggle T4/P100
    accumulation_steps = 1       # was 4; effective batch stays 4
    target_size = (512, 512)
    freeze_bn = True             # thesis invariant; try False with batch>=4

    # ---- Supervised loss (ablation): 'sum' | 'normalized' | 'adaptive' ----
    loss_mode = "adaptive"
    loss_bce_weight = 0.5        # used by 'sum' and 'normalized'
    loss_dice_weight = 0.5       # used by 'sum' and 'normalized'
    loss_min_bce_weight = 0.2    # 'adaptive' only
    loss_max_bce_weight = 0.8    # 'adaptive' only
    loss_sharpness = 2.0         # 'adaptive' only: higher = BCE keeps the lead longer

    # ---- Augmentation (rotate-and-crop version, no border artifacts) ----
    augment = True
    augment_prob = 0.6
    augment_factor = 3
    rotation_range = (-30, 30)
    scale_range = (0.85, 1.15)

    # ---- Runtime / outputs ----
    num_workers = 2
    save_dir = "/kaggle/working/checkpoints/final"
    panels_dir = "/kaggle/working/output/test_panels"
    unlabeled_out_dir = "/kaggle/working/data/unlabeled_predictions"
    threshold = 0.5
    title_tag = "UNITPathSSL + DWT-FPN"

cfg = Config()
for d in (cfg.save_dir, cfg.panels_dir):
    os.makedirs(d, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert os.path.isdir(cfg.train_img_dir), f"Image dir not found: {cfg.train_img_dir}"
print("Device:", device, "| Train images:", len(os.listdir(cfg.train_img_dir)))

## Stage A — Synthetic Pseudo Masks (`generate_pseudo_masks.py`)
Creates binary polygon masks that mimic nuclei layouts: 2–4 random "glandular" cluster centers, 15–115 nuclei per mask (mostly clustered near glands, rest scattered), ~70% oblong (randomly oriented ellipses) and the rest roundish. Every nucleus is kept at least 1 background pixel apart from its neighbours so touching nuclei still show a clear boundary, as in the real MoNuSeg masks. These form **Domain X** for the unpaired translation pretraining — no real annotations needed.

In [ ]:
# ============ 3. Pseudo mask generator ============
def generate_polygon_mask(size=(512, 512), num_nuclei=None, min_radius=4, max_radius=9,
                          num_vertices=48, nuclei_range=(15, 115),
                          oblong_prob=0.7, elongation_range=(1.4, 2.6), boundary_px=1):
    """Distributiveness is drawn per mask from a Beta(1, 2.5) scaled to [0, 2] (skewed towards 0):
    ~0 -> tight clumps, strong clustering;  ~2 -> evenly spread, widely separated nuclei.
    num_nuclei=None -> count drawn uniformly from nuclei_range (default 15..115).
    oblong_prob      -> fraction of nuclei drawn as randomly oriented ellipses (axis ratio from elongation_range).
    boundary_px      -> every nucleus is separated from its neighbours by at least this many background pixels,
                        like the clear inter-nuclear boundary in real MoNuSeg masks (no merged blobs).
    """
    h, w = size
    d = float(2.0 * np.random.beta(1.0, 2.5))     # in [0, 2], mean ~0.57, mostly low
    if num_nuclei is None:
        actual_nuclei = int(np.random.randint(nuclei_range[0], nuclei_range[1] + 1))
    else:
        actual_nuclei = int(num_nuclei)

    # Random glandular clusters (centers may sit near the edges too)
    num_glands = np.random.randint(2, 5)
    gland_centers = np.column_stack([
        np.random.randint(0, w, num_glands),
        np.random.randint(0, h, num_glands),
    ])

    placed_centers = []
    cluster_prob = max(0.0, 0.7 - 0.4 * d)  # low d -> mostly clustered; d >= 1.75 -> fully scattered
    spread_lo, spread_hi = 10 + 30 * d, 40 + 90 * d  # low d -> tight clusters

    mask = np.zeros(size, dtype=np.uint8)
    occupied = np.zeros(size, dtype=np.uint8)    # mask dilated by boundary_px: the 'no-go' zone for new nuclei
    ring_k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * boundary_px + 1, 2 * boundary_px + 1))
    t = np.linspace(0, 2 * np.pi, num_vertices, endpoint=False)

    def sample_center(min_sep):
        for _attempt in range(10):
            if np.random.rand() < cluster_prob and len(gland_centers) > 0:
                g_center = gland_centers[np.random.randint(0, num_glands)]
                angle = np.random.uniform(0, 2 * np.pi)
                r_dist = np.random.uniform(spread_lo, spread_hi)
                tx = int(np.clip(g_center[0] + r_dist * np.cos(angle), 0, w - 1))
                ty = int(np.clip(g_center[1] + r_dist * np.sin(angle), 0, h - 1))
            else:
                tx = np.random.randint(0, w)
                ty = np.random.randint(0, h)
            if all((tx - px) ** 2 + (ty - py) ** 2 >= min_sep ** 2 for px, py in placed_centers):
                return tx, ty
        return None

    def nucleus_outline(cx, cy, a_ax, b_ax, theta, r_mod):
        # Smooth closed contour: ellipse radius modulated by low harmonics only (always convex-looking)
        ex, ey = a_ax * np.cos(t) * r_mod, b_ax * np.sin(t) * r_mod
        xs = cx + ex * np.cos(theta) - ey * np.sin(theta)
        ys = cy + ex * np.sin(theta) + ey * np.cos(theta)
        return np.stack([xs, ys], 1).round().astype(np.int32)

    for _ in range(actual_nuclei):
        base_sep = max_radius * (0.4 + 1.6 * d)
        min_sep = base_sep * np.random.uniform(0.5, 1.5)

        # Shape: ellipse (oblong with prob oblong_prob, otherwise near-circular) whose radius is modulated by
        # 2-3 low-frequency harmonics -> smooth, gently curved, slightly asymmetric outline like a real nucleus.
        if np.random.rand() < oblong_prob:
            elong = np.random.uniform(*elongation_range)
        else:
            elong = np.random.uniform(1.0, 1.25)
        a_ax = np.random.uniform(min_radius, max_radius) * np.sqrt(elong)   # semi-major
        b_ax = a_ax / elong                                                 # semi-minor
        theta = np.random.uniform(0, np.pi)
        r_mod = np.ones_like(t)
        for k in (2, 3):                                        # only low harmonics: no wiggles / jaggies
            r_mod += np.random.uniform(0.0, 0.06) * np.cos(k * t + np.random.uniform(0, 2 * np.pi))

        # Overlap handling: never carve a nucleus (that leaves sharp cusps as if one lay on top of another).
        # Instead re-sample the position, shrinking the nucleus a little each time, until its WHOLE smooth
        # outline fits outside every neighbour's boundary ring. All nuclei therefore lie in one plane with
        # complete convex contours and >= boundary_px of background between them.
        placed = False
        shrink = 1.0
        for _try in range(6):
            c = sample_center(min_sep)
            if c is None:
                break
            pts = nucleus_outline(c[0], c[1], a_ax * shrink, b_ax * shrink, theta, r_mod)
            nuc = np.zeros(size, dtype=np.uint8)
            cv2.fillPoly(nuc, [pts], 255)
            if cv2.countNonZero(nuc) >= 6 and not np.any(nuc & occupied):
                placed_centers.append(c)
                mask |= nuc
                occupied |= cv2.dilate(nuc, ring_k)
                placed = True
                break
            shrink *= 0.9
        if not placed:
            continue   # no room for this nucleus; skip it
    return mask


# Preview 3 pseudo masks (always safe to run)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax in axes:
    ax.imshow(generate_polygon_mask(size=cfg.pseudo_mask_size), cmap="gray")
    ax.axis("off")
plt.suptitle("Synthetic pseudo masks (Domain X)"); plt.show()

if cfg.RUN_PRETRAINING:
    os.makedirs(cfg.pseudo_mask_dir, exist_ok=True)
    print(f"Generating {cfg.num_pseudo_masks} masks in {cfg.pseudo_mask_dir}...")
    for idx in range(cfg.num_pseudo_masks):
        m = generate_polygon_mask(size=cfg.pseudo_mask_size)
        cv2.imwrite(os.path.join(cfg.pseudo_mask_dir, f"pseudo_mask_{idx:05d}.png"), m)
    print("Pseudo mask generation complete.")

In [ ]:
# ============ 3b. Sanity tests for generate_polygon_mask ============
def test_generate_polygon_mask():
    # shape & dtype
    for size in [(512, 512), (256, 256)]:
        m = generate_polygon_mask(size=size)
        assert m.shape == size, f"expected shape {size}, got {m.shape}"
        assert m.dtype == np.uint8, f"expected uint8, got {m.dtype}"
    print("PASS shape & dtype")

    # strictly binary (only 0 and 255)
    m = generate_polygon_mask()
    vals = set(np.unique(m).tolist())
    assert vals <= {0, 255}, f"non-binary values found: {vals}"
    print("PASS binary values (only 0 and 255)")

    # foreground coverage: never empty, never mostly white
    ratios = [(generate_polygon_mask() == 255).mean() for _ in range(5)]
    mean_ratio = float(np.mean(ratios))
    assert 0.01 < mean_ratio < 0.5, f"suspicious foreground ratio {mean_ratio:.3f}"
    print(f"PASS foreground coverage (mean {mean_ratio:.1%})")

    # connected components: nuclei never merge now, so blob count == placed nuclei (some may be skipped)
    m = generate_polygon_mask()
    num_labels, _, stats, _ = cv2.connectedComponentsWithStats(m)
    n_blobs = num_labels - 1  # minus background
    assert 5 <= n_blobs <= 115, f"unexpected component count {n_blobs}"
    areas = stats[1:, cv2.CC_STAT_AREA]
    assert areas.min() >= 3, f"component too small: {areas.min()} px"
    print(f"PASS components: {n_blobs} blobs, area range {areas.min()}-{areas.max()} px")

    # boundary: a 1px dilation must not create any NEW merges beyond adjacency -> every nucleus is a
    # separate component in the raw mask (no two nuclei share a pixel)
    k = np.ones((3, 3), np.uint8)
    n_dil = cv2.connectedComponentsWithStats(cv2.dilate(m, k))[0] - 1
    assert n_dil <= n_blobs, "dilation cannot increase component count"
    print(f"PASS boundary ({n_blobs} separate nuclei; {n_blobs - n_dil} neighbour pairs sit exactly 1px apart)")

    # randomness: two consecutive calls must differ
    assert not np.array_equal(generate_polygon_mask(), generate_polygon_mask()), \
        "two calls produced identical masks"
    print("PASS randomness (consecutive masks differ)")

    print("\nAll pseudo-mask tests passed.")

test_generate_polygon_mask()

## Model Components (`loss.py`, `DWTFusion.py`, `ModifiedFPN.py`, `backbone.py`, `seg_head.py`, `generator_g.py`)
Everything the pipeline shares, defined once:
- **Losses:** BCE+Dice for supervised fine-tuning; `UNITPathSSLLoss` (adversarial + cycle + ISG) for pretraining.
- **DWT-FPN stack:** `DWTEntropyFusion` → `ModifiedFPN` → `SegModel` (this *is* the Segmentor **S** during pretraining AND the downstream model).
- **`CoModulatedGenerator` G:** pseudo mask + noise → synthetic H&E image (StyleGAN2-style weight demodulation).

In [ ]:
# ============ 4. Losses (loss.py) ============
bce_loss = nn.BCEWithLogitsLoss()

def dice_loss(pred, target, smooth=1.0):
    pred = torch.sigmoid(pred).view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    return 1.0 - ((2.0 * intersection + smooth) / (pred.sum() + target.sum() + smooth))

def total_loss(pred, target, mode="adaptive", bce_weight=0.5, dice_weight=0.5,
               min_bce_weight=0.2, max_bce_weight=0.8, sharpness=2.0, return_parts=False):
    """Downstream Supervised Fine-Tuning Loss. `mode` selects the formulation (for ablation):

    "sum"        : bce_weight * BCE + dice_weight * Dice            (original; unbounded)
    "normalized" : bce_weight * (1 - exp(-BCE)) + dice_weight * Dice (both terms in [0,1]; fixed weights)
    "adaptive"   : w * (1 - exp(-BCE)) + (1 - w) * Dice, where
                   w = clamp(1 - exp(-sharpness * BCE.detach()), min_bce_weight, max_bce_weight)
                   -> BCE dominates early (high BCE), Dice dominates late (low BCE)

    return_parts=True also returns a dict of detached components for logging.
    """
    bce = bce_loss(pred, target)
    dice = dice_loss(pred, target)

    if mode == "sum":
        w_bce, w_dice, bce_term = bce_weight, dice_weight, bce
    elif mode == "normalized":
        w_bce, w_dice, bce_term = bce_weight, dice_weight, 1.0 - torch.exp(-bce)
    elif mode == "adaptive":
        w_bce = torch.clamp(1.0 - torch.exp(-sharpness * bce.detach()), min_bce_weight, max_bce_weight)
        w_dice, bce_term = 1.0 - w_bce, 1.0 - torch.exp(-bce)
    else:
        raise ValueError(f"Unknown loss mode '{mode}'. Use 'sum', 'normalized' or 'adaptive'.")

    total = w_bce * bce_term + w_dice * dice
    if return_parts:
        return total, {"bce": bce.item(), "dice_loss": dice.item(),
                       "w_bce": float(w_bce), "w_dice": float(w_dice)}
    return total

def supervised_loss(pred, target):
    """total_loss bound to the ablation settings in cfg (used by training & test cells)."""
    return total_loss(pred, target, mode=cfg.loss_mode, bce_weight=cfg.loss_bce_weight,
                      dice_weight=cfg.loss_dice_weight, min_bce_weight=cfg.loss_min_bce_weight,
                      max_bce_weight=cfg.loss_max_bce_weight, sharpness=cfg.loss_sharpness)


def r1_penalty(d_out, real_in):
    """R1: E[||grad_x D(x)||^2] on REAL samples (real_in must have requires_grad=True)."""
    grad, = torch.autograd.grad(outputs=d_out.sum(), inputs=real_in, create_graph=True)
    return grad.pow(2).reshape(grad.shape[0], -1).sum(1).mean()

class UNITPathSSLLoss(nn.Module):
    """L = L_GAN(G,D_G) + l1*L_GAN(S,D_S) + l2*L_cyc(G,S) + l3*L_seg(S)
    Eq.1  G/D_G : non-saturating logistic GAN + R1(gamma_g) on real tissue y
    Eq.2  S/D_S : least-squares GAN        + R1(gamma_s) on real mask x ;  D_S fake = S(y)
    Eq.3  L_cyc : ||S(G(x)) - x||_1 + ||G(S(y)) - y||_1  (both directions)"""
    def __init__(self, lambda1=2.0, lambda2=10.0, lambda3=2.0, gamma_g=10.0, gamma_s=10.0):
        super().__init__()
        self.lambda1, self.lambda2, self.lambda3 = lambda1, lambda2, lambda3
        self.gamma_g, self.gamma_s = gamma_g, gamma_s
        self.l1_loss = nn.L1Loss()

    def d_g_loss(self, d_real, d_fake, real_y=None):            # Eq.1 (D_G)
        loss = F.softplus(-d_real).mean() + F.softplus(d_fake).mean()
        r1 = torch.tensor(0.0, device=d_real.device)
        if real_y is not None and self.gamma_g > 0:
            r1 = r1_penalty(d_real, real_y); loss = loss + 0.5 * self.gamma_g * r1
        return loss, {"d_g_adv": loss.item(), "d_g_r1": r1.item()}

    def d_s_loss(self, d_real, d_fake, real_x=None):            # Eq.2 (D_S)
        loss = 0.5 * ((d_real - 1.0) ** 2).mean() + 0.5 * (d_fake ** 2).mean()
        r1 = torch.tensor(0.0, device=d_real.device)
        if real_x is not None and self.gamma_s > 0:
            r1 = r1_penalty(d_real, real_x); loss = loss + 0.5 * self.gamma_s * r1
        return loss, {"d_s_adv": loss.item(), "d_s_r1": r1.item()}

    def forward(self, d_g_fake, d_s_fake, x_rec, real_x, y_rec, real_y, seg_loss_dict=None):  # G + S
        loss_adv_g = F.softplus(-d_g_fake).mean()                # -log D_G(G(x))
        loss_adv_s = 0.5 * ((d_s_fake - 1.0) ** 2).mean()        # 1/2 (D_S(S(y)) - 1)^2
        loss_cyc_x = self.l1_loss(x_rec, real_x)                 # S(G(x)) ~ x
        loss_cyc_y = self.l1_loss(y_rec, real_y)                 # G(S(y)) ~ y
        loss_isg = torch.tensor(0.0, device=x_rec.device)
        if seg_loss_dict is not None:
            loss_isg = sum(loss for loss in seg_loss_dict.values())
        total = loss_adv_g + self.lambda1 * loss_adv_s + self.lambda2 * (loss_cyc_x + loss_cyc_y) + self.lambda3 * loss_isg
        return total, {"loss_adv_g": loss_adv_g.item(), "loss_adv_s": loss_adv_s.item(),
                       "loss_cyc_x": loss_cyc_x.item(), "loss_cyc_y": loss_cyc_y.item(), "loss_isg": loss_isg.item()}

In [ ]:
# ============ 4b. Visualize mask -> binary target (interactive threshold) ============
# Pixels >= threshold become 255 (foreground), pixels < threshold become 0.
# Move the slider to observe how the threshold changes the binary mask.
from ipywidgets import interact, IntSlider

mask_files = sorted(glob.glob(os.path.join(cfg.train_mask_dir, "*.png")))
if mask_files:
    raw = cv2.imread(mask_files[0], cv2.IMREAD_GRAYSCALE)
else:  # fallback: use a generated pseudo mask
    raw = generate_polygon_mask(size=(512, 512))

print(f"raw unique values ({len(np.unique(raw))}): {np.unique(raw)[:12]}{' ...' if len(np.unique(raw)) > 12 else ''}")

def show_threshold(threshold=128):
    binary = np.where(raw >= threshold, 255, 0).astype(np.uint8)
    changed = (raw != binary)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(raw, cmap="gray", vmin=0, vmax=255)
    axes[0].set_title(f"Raw mask (min={raw.min()}, max={raw.max()})")
    axes[1].imshow(binary, cmap="gray", vmin=0, vmax=255)
    axes[1].set_title(f"Binary (>= {threshold} -> 255, < {threshold} -> 0)")
    axes[2].imshow(changed, cmap="hot")
    axes[2].set_title(f"Changed pixels: {changed.sum()} ({changed.mean():.2%})")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout(); plt.show()
    print(f"foreground: {(binary == 255).mean():.2%} of image")

interact(show_threshold, threshold=IntSlider(value=128, min=0, max=255, step=1, description="Threshold"));

In [ ]:
# ============ 5. DWT Entropy Fusion (DWTFusion.py) ============
class DWTEntropyFusion(nn.Module):
    """Entropy-guided spatial attention fusion for Wavelet high-frequency components."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv_reduce = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.attn_conv = nn.Sequential(
            nn.Conv2d(in_channels * 3, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 3, kernel_size=1))

    def local_entropy(self, x):
        p = torch.abs(x)
        p = F.avg_pool2d(p, kernel_size=3, stride=1, padding=1)
        p = p / (p.sum(dim=(2, 3), keepdim=True) + 1e-8)
        ent = -p * torch.log(p + 1e-8)
        return torch.mean(ent, dim=1, keepdim=True)

    def forward(self, LH, HL, HH):
        LH = (LH - LH.mean(dim=(2, 3), keepdim=True)) / (LH.std(dim=(2, 3), keepdim=True) + 1e-6)
        HL = (HL - HL.mean(dim=(2, 3), keepdim=True)) / (HL.std(dim=(2, 3), keepdim=True) + 1e-6)
        HH = (HH - HH.mean(dim=(2, 3), keepdim=True)) / (HH.std(dim=(2, 3), keepdim=True) + 1e-6)

        entropy_weights = torch.cat([self.local_entropy(LH), self.local_entropy(HL),
                                     self.local_entropy(HH)], dim=1)
        attn = self.attn_conv(torch.cat([LH, HL, HH], dim=1))
        weights = F.softmax(entropy_weights + attn, dim=1)
        F_out = weights[:, 0:1] * LH + weights[:, 1:2] * HL + weights[:, 2:3] * HH
        return self.conv_reduce(F_out)

In [ ]:
# ============ 6. Modified FPN (ModifiedFPN.py) ============
class ModifiedFPN(nn.Module):
    """FPN with GroupNorm (32 groups), DWT Entropy Fusion, and P6 level."""
    def __init__(self, in_channels=(256, 512, 1024, 2048), out_channels=256, num_groups=32):
        super().__init__()
        if len(in_channels) != 4:
            raise ValueError("ModifiedFPN requires exactly 4 backbone feature channels")

        def lat(c_in):
            return nn.Sequential(nn.Conv2d(c_in, out_channels, kernel_size=1),
                                 nn.GroupNorm(num_groups, out_channels))
        self.lat2, self.lat3 = lat(in_channels[0]), lat(in_channels[1])
        self.lat4, self.lat5 = lat(in_channels[2]), lat(in_channels[3])

        def smooth():
            return nn.Sequential(
                nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
                nn.GroupNorm(num_groups, out_channels),
                nn.LeakyReLU(0.2, inplace=True))
        self.smooth2, self.smooth3 = smooth(), smooth()
        self.smooth4, self.smooth5 = smooth(), smooth()

        self.p6 = nn.MaxPool2d(kernel_size=1, stride=2, padding=0)

        self.dwt2_fuse = DWTEntropyFusion(in_channels[0], out_channels)
        self.dwt3_fuse = DWTEntropyFusion(in_channels[1], out_channels)
        self.dwt4_fuse = DWTEntropyFusion(in_channels[2], out_channels)

        self.register_buffer("ll", torch.tensor([[0.5, 0.5], [0.5, 0.5]]))
        self.register_buffer("lh", torch.tensor([[-0.5, -0.5], [0.5, 0.5]]))
        self.register_buffer("hl", torch.tensor([[-0.5, 0.5], [-0.5, 0.5]]))
        self.register_buffer("hh", torch.tensor([[0.5, -0.5], [-0.5, 0.5]]))

    def dwt(self, x):
        B, C, H, W = x.shape
        if H % 2 != 0 or W % 2 != 0:
            x = F.pad(x, (0, W % 2, 0, H % 2))
        filters = torch.stack([self.ll, self.lh, self.hl, self.hh], dim=0).unsqueeze(1)
        filters = filters.repeat(C, 1, 1, 1)
        out = F.conv2d(x, filters, stride=2, groups=C)
        out = out.view(B, C, 4, out.shape[2], out.shape[3])
        return out[:, :, 1], out[:, :, 2], out[:, :, 3]

    def forward(self, C2, C3, C4, C5):
        P5_lat = self.lat5(C5)
        P4_lat = self.lat4(C4)
        P3_lat = self.lat3(C3)
        P2_lat = self.lat2(C2)

        P4 = P4_lat + F.interpolate(P5_lat, size=P4_lat.shape[2:], mode='nearest')
        P3 = P3_lat + F.interpolate(P4, size=P3_lat.shape[2:], mode='nearest')
        P2 = P2_lat + F.interpolate(P3, size=P2_lat.shape[2:], mode='nearest')

        LH2, HL2, HH2 = self.dwt(C2)
        LH3, HL3, HH3 = self.dwt(C3)
        LH4, HL4, HH4 = self.dwt(C4)

        F2 = F.interpolate(self.dwt2_fuse(LH2, HL2, HH2), size=P2.shape[2:], mode='bilinear', align_corners=False)
        F3 = F.interpolate(self.dwt3_fuse(LH3, HL3, HH3), size=P3.shape[2:], mode='bilinear', align_corners=False)
        F4 = F.interpolate(self.dwt4_fuse(LH4, HL4, HH4), size=P4.shape[2:], mode='bilinear', align_corners=False)

        P2 = self.smooth2(P2 + F2)
        P3 = self.smooth3(P3 + F3)
        P4 = self.smooth4(P4 + F4)
        P5 = self.smooth5(P5_lat)
        P6 = self.p6(P5)
        return P2, P3, P4, P5, P6

In [ ]:
# ============ 7. Backbones (backbone.py) ============
from torchvision.models import (
    resnet18, resnet34, resnet50,
    ResNet18_Weights, ResNet34_Weights, ResNet50_Weights,
    mobilenet_v2, MobileNet_V2_Weights,
    efficientnet_b0, EfficientNet_B0_Weights,
    densenet121, DenseNet121_Weights,
)


def _get_resnet_channels(resnet):
    def _layer_channels(layer):
        block = layer[-1]
        return block.bn3.num_features if hasattr(block, 'bn3') else block.bn2.num_features
    return [_layer_channels(resnet.layer1), _layer_channels(resnet.layer2),
            _layer_channels(resnet.layer3), _layer_channels(resnet.layer4)]


class ResNetBackbone(nn.Module):
    def __init__(self, variant="resnet50", pretrained=False):
        super().__init__()
        weights_map = {"resnet18": ResNet18_Weights.IMAGENET1K_V1,
                       "resnet34": ResNet34_Weights.IMAGENET1K_V1,
                       "resnet50": ResNet50_Weights.IMAGENET1K_V1}
        if variant not in weights_map:
            raise ValueError(f"Unsupported ResNet variant: {variant}")
        resnet_fn = {"resnet18": resnet18, "resnet34": resnet34, "resnet50": resnet50}[variant]
        resnet = resnet_fn(weights=weights_map[variant] if pretrained else None)

        self.stage0 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.stage1, self.stage2 = resnet.layer1, resnet.layer2
        self.stage3, self.stage4 = resnet.layer3, resnet.layer4
        self.out_channels = _get_resnet_channels(resnet)

    def forward(self, x):
        x = self.stage0(x)
        C2 = self.stage1(x); C3 = self.stage2(C2)
        C4 = self.stage3(C3); C5 = self.stage4(C4)
        return C2, C3, C4, C5


class MobileNetV2Backbone(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()
        model = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None)
        self.stage0 = nn.Sequential(model.features[0], model.features[1], model.features[2])
        self.stage1 = nn.Sequential(*model.features[3:7])
        self.stage2 = nn.Sequential(*model.features[7:14])
        self.stage3 = nn.Sequential(*model.features[14:])
        self.out_channels = [24, 32, 96, 1280]

    def forward(self, x):
        C2 = self.stage0(x); C3 = self.stage1(C2)
        C4 = self.stage2(C3); C5 = self.stage3(C4)
        return C2, C3, C4, C5


class EfficientNetBackbone(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()
        model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
        self.stage0 = nn.Sequential(model.features[0], model.features[1], model.features[2])
        self.stage1 = nn.Sequential(model.features[3])
        self.stage2 = nn.Sequential(model.features[4])
        self.stage3 = nn.Sequential(*model.features[5:])
        self.out_channels = [24, 40, 80, 1280]

    def forward(self, x):
        C2 = self.stage0(x); C3 = self.stage1(C2)
        C4 = self.stage2(C3); C5 = self.stage3(C4)
        return C2, C3, C4, C5


class DenseNetBackbone(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()
        model = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None)
        self.stage0 = nn.Sequential(*model.features[:5])
        self.stage1 = nn.Sequential(model.features[5])
        self.stage2 = nn.Sequential(model.features[6], model.features[7])
        self.stage3 = nn.Sequential(*model.features[8:])
        self.out_channels = [256, 128, 256, 1024]

    def forward(self, x):
        C2 = self.stage0(x); C3 = self.stage1(C2)
        C4 = self.stage2(C3); C5 = self.stage3(C4)
        return C2, C3, C4, C5


def build_backbone(backbone_name="resnet50", pretrained=False):
    backbone_name = backbone_name.lower()
    if backbone_name == "resnet20":
        backbone_name = "resnet18"
    if backbone_name in {"resnet18", "resnet34", "resnet50"}:
        return ResNetBackbone(variant=backbone_name, pretrained=pretrained)
    if backbone_name == "mobilenet_v2":
        return MobileNetV2Backbone(pretrained=pretrained)
    if backbone_name == "efficientnet_b0":
        return EfficientNetBackbone(pretrained=pretrained)
    if backbone_name == "densenet121":
        return DenseNetBackbone(pretrained=pretrained)
    raise ValueError(f"Unsupported backbone: {backbone_name}")

In [ ]:
# ============ 8. Segmentation model S (seg_head.py) ============
class SegModel(nn.Module):
    """Backbone + DWT-FPN + multi-scale summation decoder. Used as Segmentor S in
    pretraining AND as the downstream fine-tuned model."""
    def __init__(self, backbone_name="resnet50", pretrained=True, fpn_out_channels=256, num_classes=1):
        super().__init__()
        self.backbone_name = backbone_name
        self.backbone = build_backbone(backbone_name, pretrained=pretrained)
        self.fpn = ModifiedFPN(in_channels=self.backbone.out_channels, out_channels=fpn_out_channels)

        self.decoder_conv = nn.Sequential(
            nn.Conv2d(fpn_out_channels, fpn_out_channels // 2, kernel_size=3, padding=1),
            nn.InstanceNorm2d(fpn_out_channels // 2, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(fpn_out_channels // 2, fpn_out_channels // 4, kernel_size=3, padding=1),
            nn.InstanceNorm2d(fpn_out_channels // 4, affine=True),
            nn.LeakyReLU(0.2, inplace=True))
        self.final_upsample = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(fpn_out_channels // 4, fpn_out_channels // 4, kernel_size=3, padding=1),
            nn.InstanceNorm2d(fpn_out_channels // 4, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(fpn_out_channels // 4, num_classes, kernel_size=3, padding=1))

    def extract_features(self, x):
        C2, C3, C4, C5 = self.backbone(x)
        return self.fpn(C2, C3, C4, C5)

    def forward(self, x):
        P2, P3, P4, P5, P6 = self.extract_features(x)
        target_size = P2.shape[2:]
        u3 = F.interpolate(P3, size=target_size, mode="bilinear", align_corners=False)
        u4 = F.interpolate(P4, size=target_size, mode="bilinear", align_corners=False)
        u5 = F.interpolate(P5, size=target_size, mode="bilinear", align_corners=False)
        u6 = F.interpolate(P6, size=target_size, mode="bilinear", align_corners=False)
        fused = P2 + u3 + u4 + u5 + u6
        return self.final_upsample(self.decoder_conv(fused))

In [ ]:
# ============ 8b. DWT pipeline on ONE image (no backbone, no FPN, no upsampling) ============
# Feeds a single RGB tile straight through: Haar DWT -> DWTEntropyFusion (pre-upsample output),
# and compares the fused high-frequency map against the raw image + a classical edge detector.
# Runs on CPU, ~1s. Change DWT_TEST_IMAGE to inspect a different file.
DWT_TEST_IMAGE = None   # e.g. "/kaggle/input/.../TCGA-A7-A13E-01Z-00-DX1.tif"; None -> first training image

def _load_dwt_test_image(path=None, size=cfg.target_size):
    if path is None:
        cands = sorted(glob.glob(os.path.join(cfg.train_img_dir, "*")))
        if not cands:
            raise FileNotFoundError(f"No images in {cfg.train_img_dir}")
        path = cands[0]
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, size, interpolation=cv2.INTER_AREA)
    t = torch.from_numpy(img).permute(2, 0, 1).float().unsqueeze(0) / 255.0   # [1,3,H,W] in [0,1]
    return img, t, os.path.basename(path)

def run_dwt_pipeline(x, fusion=None, seed=0):
    """x: [1,C,H,W]. Returns dict with the 3 Haar sub-bands, entropy prior, softmax weights and the
    fused map EXACTLY as DWTEntropyFusion emits it (before ModifiedFPN's bilinear upsample)."""
    torch.manual_seed(seed)
    C = x.shape[1]
    if fusion is None:
        fusion = DWTEntropyFusion(C, 1)      # out_channels=1 so the result is a single viewable map
    fusion.eval()
    with torch.no_grad():
        haar = ModifiedFPN(in_channels=(C, C, C, C), out_channels=32)   # only its fixed Haar buffers are used
        LH, HL, HH = haar.dwt(x)
        # replicate the internals so we can look at each stage
        norm = lambda b: (b - b.mean(dim=(2, 3), keepdim=True)) / (b.std(dim=(2, 3), keepdim=True) + 1e-6)
        LHn, HLn, HHn = norm(LH), norm(HL), norm(HH)
        ent = torch.cat([fusion.local_entropy(LHn), fusion.local_entropy(HLn), fusion.local_entropy(HHn)], 1)
        attn = fusion.attn_conv(torch.cat([LHn, HLn, HHn], 1))
        w = F.softmax(ent + attn, dim=1)
        fused = fusion(LH, HL, HH)           # == conv_reduce(w0*LH + w1*HL + w2*HH)
    return dict(LH=LH, HL=HL, HH=HH, entropy=ent, attn=attn, weights=w, fused=fused)

def _to_img(t):  # [1,C,h,w] -> [h,w] magnitude normalised to [0,1]
    m = t[0].abs().mean(0)
    return ((m - m.min()) / (m.max() - m.min() + 1e-8)).cpu().numpy()

img_np, img_t, img_name = _load_dwt_test_image(DWT_TEST_IMAGE)
out = run_dwt_pipeline(img_t)

# classical reference: Sobel magnitude on the grayscale image, downsampled to DWT resolution
gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY).astype(np.float32)
sob = np.hypot(cv2.Sobel(gray, cv2.CV_32F, 1, 0), cv2.Sobel(gray, cv2.CV_32F, 0, 1))
sob = cv2.resize(sob, out["fused"].shape[-2:][::-1], interpolation=cv2.INTER_AREA)
sob = (sob - sob.min()) / (sob.max() - sob.min() + 1e-8)
fused_img = _to_img(out["fused"])
corr = np.corrcoef(fused_img.ravel(), sob.ravel())[0, 1]

print(f"image: {img_name} | input {tuple(img_t.shape)} -> sub-bands {tuple(out['LH'].shape)} -> fused {tuple(out['fused'].shape)}")
print(f"mean softmax weight per band  LH={out['weights'][0,0].mean():.3f}  HL={out['weights'][0,1].mean():.3f}  HH={out['weights'][0,2].mean():.3f}")
print(f"Pearson corr(fused, Sobel) = {corr:.3f}   (untrained attention; >0.4 means the Haar path alone already tracks edges)")

fig, ax = plt.subplots(2, 4, figsize=(18, 9))
ax[0,0].imshow(img_np);                        ax[0,0].set_title("input RGB")
ax[0,1].imshow(_to_img(out["LH"]), cmap="gray"); ax[0,1].set_title("LH (horizontal edges)")
ax[0,2].imshow(_to_img(out["HL"]), cmap="gray"); ax[0,2].set_title("HL (vertical edges)")
ax[0,3].imshow(_to_img(out["HH"]), cmap="gray"); ax[0,3].set_title("HH (diagonal/texture)")
ax[1,0].imshow(out["entropy"][0].mean(0).cpu(), cmap="magma"); ax[1,0].set_title("entropy prior (mean of 3)")
ax[1,1].imshow(out["weights"][0].permute(1, 2, 0).cpu());       ax[1,1].set_title("softmax weights (R=LH,G=HL,B=HH)")
ax[1,2].imshow(fused_img, cmap="gray");         ax[1,2].set_title("DWTEntropyFusion output (pre-upsample)")
ax[1,3].imshow(sob, cmap="gray");               ax[1,3].set_title(f"Sobel reference  corr={corr:.2f}")
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ============ 8c. Unit tests + parameter probe for DWTEntropyFusion / ModifiedFPN.dwt ============
# Pure checks on the maths (no data, no training). Re-run after any change to cells 5/6.
def test_dwt_fusion():
    torch.manual_seed(0)
    C, H, W = 8, 32, 32
    fpn = ModifiedFPN(in_channels=(C, C, C, C), out_channels=32)   # out_channels must be divisible by GroupNorm groups (32)
    fus = DWTEntropyFusion(C, 32)

    # 1. Haar filters are orthogonal & energy-preserving: sum of squares of the 4 sub-bands == input energy
    x = torch.randn(2, C, H, W)
    filt = torch.stack([fpn.ll, fpn.lh, fpn.hl, fpn.hh]).unsqueeze(1).repeat(C, 1, 1, 1)
    bands = F.conv2d(x, filt, stride=2, groups=C)
    assert torch.allclose(bands.pow(2).sum(), x.pow(2).sum(), rtol=1e-4), "Haar not energy preserving"

    # 2. dwt() returns exactly (LH, HL, HH) at half resolution and ignores LL
    LH, HL, HH = fpn.dwt(x)
    assert LH.shape == (2, C, H // 2, W // 2) and HL.shape == LH.shape and HH.shape == LH.shape
    const = torch.ones(1, C, H, W) * 3.0                      # a flat image has NO high-frequency content
    assert all(b.abs().max() < 1e-6 for b in fpn.dwt(const)), "flat image should give zero high-freq"

    # 3. Orientation selectivity: horizontal step edge -> LH responds, HL does not (and vice versa)
    #    (edge at an ODD index so a stride-2 2x2 Haar block actually straddles it; an even index falls on a
    #     block boundary and correctly yields zero high-frequency response)
    hstep = torch.zeros(1, C, H, W); hstep[..., H // 2 + 1:, :] = 1.0
    vstep = torch.zeros(1, C, H, W); vstep[..., :, W // 2 + 1:] = 1.0
    lh_h, hl_h, _ = fpn.dwt(hstep); lh_v, hl_v, _ = fpn.dwt(vstep)
    assert lh_h.abs().sum() > 0 and hl_h.abs().max() < 1e-6, "LH should detect horizontal edges only"
    assert hl_v.abs().sum() > 0 and lh_v.abs().max() < 1e-6, "HL should detect vertical edges only"

    # 4. Odd sizes are padded, not crashed
    o = fpn.dwt(torch.randn(1, C, 31, 33))[0]
    assert o.shape[-2:] == (16, 17), o.shape

    # 5. Entropy prior: flat band -> uniform p -> maximum entropy; peaky band -> lower entropy
    flat = torch.ones(1, C, 16, 16); peaky = torch.zeros(1, C, 16, 16); peaky[..., 8, 8] = 100.0
    assert fus.local_entropy(flat).mean() > fus.local_entropy(peaky).mean(), "entropy ordering wrong"

    # 6. Softmax weights sum to 1 per pixel; fused output has the requested channel count
    with torch.no_grad():
        n = lambda b: (b - b.mean(dim=(2, 3), keepdim=True)) / (b.std(dim=(2, 3), keepdim=True) + 1e-6)
        ent = torch.cat([fus.local_entropy(n(LH)), fus.local_entropy(n(HL)), fus.local_entropy(n(HH))], 1)
        w = F.softmax(ent + fus.attn_conv(torch.cat([n(LH), n(HL), n(HH)], 1)), 1)
        assert torch.allclose(w.sum(1), torch.ones_like(w[:, 0]), atol=1e-5)
        out = fus(LH, HL, HH)
        assert out.shape == (2, 32, H // 2, W // 2), out.shape

    # 7. Gradients reach the learnable parts (attention + reduce) and the input
    xg = torch.randn(1, C, H, W, requires_grad=True)
    fus(*fpn.dwt(xg)).sum().backward()
    assert xg.grad is not None and xg.grad.abs().sum() > 0
    assert all(p.grad is not None for p in fus.parameters()), "some fusion params received no gradient"

    # 8. ModifiedFPN end-to-end shapes with a fake ResNet50 pyramid
    fpn50 = ModifiedFPN(in_channels=(256, 512, 1024, 2048), out_channels=256)
    C2, C3, C4, C5 = (torch.randn(1, 256, 128, 128), torch.randn(1, 512, 64, 64),
                      torch.randn(1, 1024, 32, 32), torch.randn(1, 2048, 16, 16))
    with torch.no_grad():
        P2, P3, P4, P5, P6 = fpn50(C2, C3, C4, C5)
    assert [p.shape[-1] for p in (P2, P3, P4, P5, P6)] == [128, 64, 32, 16, 8]
    assert all(p.shape[1] == 256 for p in (P2, P3, P4, P5, P6))

    n_dwt = sum(p.numel() for m in (fpn50.dwt2_fuse, fpn50.dwt3_fuse, fpn50.dwt4_fuse) for p in m.parameters())
    n_all = sum(p.numel() for p in fpn50.parameters())
    print(f"All 8 DWT/fusion tests passed.  DWT branch params: {n_dwt:,} / FPN total {n_all:,} ({100*n_dwt/n_all:.1f}%)")

test_dwt_fusion()

# ---- Probe: how much does the DWT branch actually contribute in a model? ----
# Call after fine-tuning (or on an untrained model) to see |F| vs |P| per level and the band usage.
@torch.no_grad()
def probe_dwt_contribution(model, x):
    """x: [B,3,H,W] normalised image batch. Prints relative energy of the DWT residual F_k vs the FPN
    path P_k at each level, plus the mean softmax weight per band. Near-zero ratio == branch ignored;
    near-uniform weights (~0.33) == attention not discriminating."""
    model.eval(); fpn = model.fpn
    C2, C3, C4, C5 = model.backbone(x)
    P5 = fpn.lat5(C5); P4 = fpn.lat4(C4) + F.interpolate(P5, size=C4.shape[2:], mode="nearest")
    P3 = fpn.lat3(C3) + F.interpolate(P4, size=C3.shape[2:], mode="nearest")
    P2 = fpn.lat2(C2) + F.interpolate(P3, size=C2.shape[2:], mode="nearest")
    n = lambda b: (b - b.mean(dim=(2, 3), keepdim=True)) / (b.std(dim=(2, 3), keepdim=True) + 1e-6)
    for name, Ck, Pk, fus in (("P2", C2, P2, fpn.dwt2_fuse), ("P3", C3, P3, fpn.dwt3_fuse), ("P4", C4, P4, fpn.dwt4_fuse)):
        LH, HL, HH = fpn.dwt(Ck)
        Fk = F.interpolate(fus(LH, HL, HH), size=Pk.shape[2:], mode="bilinear", align_corners=False)
        ent = torch.cat([fus.local_entropy(n(LH)), fus.local_entropy(n(HL)), fus.local_entropy(n(HH))], 1)
        w = F.softmax(ent + fus.attn_conv(torch.cat([n(LH), n(HL), n(HH)], 1)), 1).mean(dim=(0, 2, 3))
        print(f"{name}: |F|/|P| = {Fk.norm() / (Pk.norm() + 1e-8):.3f} | band weights LH={w[0]:.2f} HL={w[1]:.2f} HH={w[2]:.2f}")

# Example (untrained model, ImageNet backbone):
# _m = SegModel(cfg.backbone, pretrained=True).to(device)
# probe_dwt_contribution(_m, img_t.to(device))   # img_t from cell 8b (add ImageNet normalisation for a real check)

In [ ]:
# ============ 9. Co-modulated Generator G (generator_g.py) ============
class ModulatedConv2d(nn.Module):
    """Weight-demodulation conv (StyleGAN2/CoModGAN): per-sample style modulates weights."""
    def __init__(self, in_channels, out_channels, kernel_size=3, style_dim=512):
        super().__init__()
        self.in_channels, self.out_channels = in_channels, out_channels
        self.kernel_size, self.padding = kernel_size, kernel_size // 2
        self.weight = nn.Parameter(torch.randn(out_channels, in_channels, kernel_size, kernel_size))
        self.affine = nn.Linear(style_dim, in_channels)

    def forward(self, x, style):
        B, C, H, W = x.shape
        s = self.affine(style).view(B, 1, self.in_channels, 1, 1) + 1.0
        w = self.weight.unsqueeze(0) * s
        d = torch.rsqrt(w.pow(2).sum(dim=[2, 3, 4], keepdim=True) + 1e-8)
        w_demod = (w * d).view(B * self.out_channels, self.in_channels,
                               self.kernel_size, self.kernel_size)
        x = x.view(1, B * C, H, W)
        out = F.conv2d(x, w_demod, padding=self.padding, groups=B)
        return out.view(B, self.out_channels, H, W)


class CoModulatedGenerator(nn.Module):
    """G: pseudo mask x + noise z -> synthetic H&E image y_hat in [-1, 1]."""
    def __init__(self, in_channels=1, out_channels=3, z_dim=512, style_dim=512, enc_dim=1024, mapping_layers=8):
        super().__init__()
        self.z_dim, self.style_dim = z_dim, style_dim
        cond_dim = enc_dim + style_dim   # [E(x), w] -> per-layer affine A gives s (Eq. 4)

        layers = []                      # Mapping network M: 8 FC layers (StyleGAN2)
        for _ in range(mapping_layers):
            layers += [nn.Linear(z_dim if not layers else style_dim, style_dim), nn.LeakyReLU(0.2, inplace=True)]
        self.mapping_net = nn.Sequential(*layers)

        self.enc1 = nn.Conv2d(in_channels, 64, kernel_size=3, padding=1)
        self.down1 = nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1)
        self.down2 = nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1)
        self.down3 = nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1)
        self.down4 = nn.Conv2d(512, 512, kernel_size=4, stride=2, padding=1)
        self.down5 = nn.Conv2d(512, 512, kernel_size=4, stride=2, padding=1)
        self.down6 = nn.Conv2d(512, 512, kernel_size=4, stride=2, padding=1)
        self.enc_conv = nn.Conv2d(512, 512, kernel_size=3, padding=1)     # Conv3x3 at 4x4 (Fig. 3)

        self.enc_fc = nn.Sequential(nn.Flatten(), nn.Linear(512 * 4 * 4, enc_dim),   # FC 1x1024
                                    nn.LeakyReLU(0.2, inplace=True), nn.Dropout(0.5))  # dropout 0.5 -> E(x)
        self.init_proj = nn.Linear(enc_dim, 512 * 4 * 4)                  # FC -> reshape, replaces const input

        self.mod1 = ModulatedConv2d(512, 512, 3, cond_dim)
        self.up1 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.skip1 = nn.Conv2d(512, 512, kernel_size=1)
        self.mod2 = ModulatedConv2d(512, 512, 3, cond_dim)
        self.up2 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.skip2 = nn.Conv2d(512, 512, kernel_size=1)
        self.mod3 = ModulatedConv2d(512, 256, 3, cond_dim)
        self.up3 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.skip3 = nn.Conv2d(512, 256, kernel_size=1)
        self.mod4 = ModulatedConv2d(256, 128, 3, cond_dim)
        self.up4 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.skip4 = nn.Conv2d(256, 128, kernel_size=1)
        self.mod5 = ModulatedConv2d(128, 64, 3, cond_dim)
        self.up5 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.skip5 = nn.Conv2d(128, 64, kernel_size=1)
        self.mod6 = ModulatedConv2d(64, 32, 3, cond_dim)
        self.up6 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.to_rgb = nn.Conv2d(32, out_channels, kernel_size=1)

    def forward(self, mask_x, z=None):
        B = mask_x.shape[0]
        if z is None:
            z = torch.randn(B, self.z_dim, device=mask_x.device)

        e1 = F.leaky_relu(self.enc1(mask_x), 0.2)
        e2 = F.leaky_relu(self.down1(e1), 0.2)
        e3 = F.leaky_relu(self.down2(e2), 0.2)
        e4 = F.leaky_relu(self.down3(e3), 0.2)
        e5 = F.leaky_relu(self.down4(e4), 0.2)
        e6 = F.leaky_relu(self.down5(e5), 0.2)
        e7 = F.leaky_relu(self.down6(e6), 0.2)
        e7 = F.leaky_relu(self.enc_conv(e7), 0.2)

        E_x = self.enc_fc(e7)                       # conditional style E(x)  [B,1024]
        w = self.mapping_net(z)                     # stochastic style  M(z)  [B,512]
        style_s = torch.cat([E_x, w], dim=1)        # s = A(E(x), M(z)); A = each ModulatedConv's affine

        h = self.init_proj(E_x).view(B, 512, 4, 4)
        h = F.leaky_relu(self.mod1(h, style_s), 0.2)
        h = self.up1(h) + self.skip1(e6)
        h = F.leaky_relu(self.mod2(h, style_s), 0.2)
        h = self.up2(h) + self.skip2(e5)
        h = F.leaky_relu(self.mod3(h, style_s), 0.2)
        h = self.up3(h) + self.skip3(e4)
        h = F.leaky_relu(self.mod4(h, style_s), 0.2)
        h = self.up4(h) + self.skip4(e3)
        h = F.leaky_relu(self.mod5(h, style_s), 0.2)
        h = self.up5(h) + self.skip5(e2)
        h = F.leaky_relu(self.mod6(h, style_s), 0.2)
        h = self.up6(h)
        return torch.tanh(self.to_rgb(h))

## Stage B — UNITPathSSL Pretraining (`pretrain_large.py`) — optional
Unpaired translation between pseudo masks (X) and real tissue patches (Y):
- **G** (CoModulatedGenerator): mask → fake tissue; **S** (SegModel): tissue → mask.
- **Forward cycle** x → G(x) → S(G(x)) ≈ x enforced with L1; PatchGAN discriminators D_G/D_S with LSGAN loss.
- Two stages: Stage 1 (40k iters @ 4e-4), then S and D_S reinitialized for Stage 2 (25k iters @ 1e-4). Only **S's weights** are kept — they become the fine-tuning initialization.

⚠️ Needs `cfg.tissue_dir` pointing at unlabeled H&E patches, and the full run exceeds a single Kaggle session — reduce `stage1_iters`/`stage2_iters` for demos, or skip and use ImageNet init.

In [ ]:
# ============ 10. Pretraining dataset + discriminator + stage runner ============
class LargeUnpairedDataset(Dataset):
    def __init__(self, mask_dir, tissue_dir, target_size=(256, 256)):
        super().__init__()
        valid_exts = (".png", ".tif", ".tiff", ".jpg", ".jpeg")
        self.mask_files = sorted([f for f in glob.glob(os.path.join(mask_dir, "**", "*.*"), recursive=True)
                                  if f.lower().endswith(valid_exts)])
        self.tissue_files = sorted([f for f in glob.glob(os.path.join(tissue_dir, "**", "*.*"), recursive=True)
                                    if f.lower().endswith(valid_exts)])
        self.target_size = tuple(target_size)
        if len(self.tissue_files) == 0 or len(self.mask_files) == 0:
            raise ValueError(f"Found {len(self.tissue_files)} tissues / {len(self.mask_files)} masks.")
        print(f"Dataset ready: {len(self.tissue_files)} tissue images, {len(self.mask_files)} pseudo masks.")

    def __len__(self):
        return max(len(self.mask_files), len(self.tissue_files))

    def __getitem__(self, idx):
        m = cv2.imread(self.mask_files[idx % len(self.mask_files)], 0)
        m = cv2.resize(m, self.target_size, interpolation=cv2.INTER_NEAREST)
        mask_t = torch.from_numpy((m > 0).astype("float32")).unsqueeze(0)

        t = cv2.imread(self.tissue_files[idx % len(self.tissue_files)])
        t = cv2.cvtColor(t, cv2.COLOR_BGR2RGB)
        t = cv2.resize(t, self.target_size, interpolation=cv2.INTER_LINEAR)
        tissue_t = (torch.from_numpy(t).permute(2, 0, 1).float() / 127.5) - 1.0  # [-1, 1]
        return mask_t, tissue_t


class PatchDiscriminator(nn.Module):
    def __init__(self, in_channels=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.GroupNorm(16, 128), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.GroupNorm(32, 256), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 1, kernel_size=4, stride=1, padding=1))

    def forward(self, x):
        return self.net(x)


class ADA(nn.Module):
    """Adaptive Discriminator Augmentation (paper Eq. 5). Applied to BOTH real y and fake G(x) before D_G.
    Differentiable per-sample augment with prob p; r_t = E[sign(D_G(real))], updated every 4 minibatches:
        p = max(p + sign(r_t - 0.6) * (N*4)/500000, 0)"""
    def __init__(self, target=0.6, interval=4, speed_denom=500000.0, p_max=1.0):
        super().__init__()
        self.target, self.interval, self.speed_denom, self.p_max = target, interval, speed_denom, p_max
        self.p = 0.0; self._sign_acc, self._n_imgs, self._calls = 0.0, 0, 0

    @torch.no_grad()
    def update(self, d_real_logits):
        self._sign_acc += torch.sign(d_real_logits).mean().item(); self._n_imgs += d_real_logits.shape[0]; self._calls += 1
        if self._calls % self.interval == 0:
            r_t = self._sign_acc / self.interval
            step = (1.0 if r_t > self.target else -1.0) * self._n_imgs / self.speed_denom
            self.p = float(min(max(self.p + step, 0.0), self.p_max))
            self._sign_acc, self._n_imgs = 0.0, 0

    def forward(self, img):
        if self.p <= 0:
            return img
        B, dev = img.shape[0], img.device
        gate = lambda: (torch.rand(B, 1, 1, 1, device=dev) < self.p).float()
        g = gate(); img = g * torch.flip(img, dims=[3]) + (1 - g) * img                     # x-flip
        g = gate(); img = g * torch.rot90(img, 1, dims=[2, 3]) + (1 - g) * img              # 90deg rot
        g = gate()
        if g.sum() > 0:                                                                     # int translation
            ms = int(0.125 * img.shape[-1]); dx, dy = [int(v) for v in torch.randint(-ms, ms + 1, (2,))]
            img = g * torch.roll(img, shifts=(dy, dx), dims=(2, 3)) + (1 - g) * img
        g = gate(); img = img + g * (torch.randn(B, 1, 1, 1, device=dev) * 0.2)             # brightness
        g = gate(); mean = img.mean(dim=(1, 2, 3), keepdim=True)                            # contrast
        img = g * ((img - mean) * torch.exp2(torch.randn(B, 1, 1, 1, device=dev) * 0.5) + mean) + (1 - g) * img
        if img.shape[1] == 3:                                                               # saturation
            g = gate(); gray = img.mean(dim=1, keepdim=True)
            img = g * ((img - gray) * torch.exp2(torch.randn(B, 1, 1, 1, device=dev)) + gray) + (1 - g) * img
        g = gate(); img = img + g * torch.randn_like(img) * (torch.rand(B, 1, 1, 1, device=dev) * 0.1)  # noise
        return img


def run_stage(stage_num, max_iters, lr, gen_g, gen_s, disc_g, disc_s, loader,
              device, accum_steps, save_dir, backbone,
              lambda1=2.0, lambda2=10.0, lambda3=0.0, gamma_g=10.0, gamma_s=10.0, r1_interval=16, ada=None):
    opt_g = optim.Adam(list(gen_g.parameters()) + list(gen_s.parameters()), lr=lr, betas=(0.0, 0.99), eps=1e-8)
    opt_d = optim.Adam(list(disc_g.parameters()) + list(disc_s.parameters()), lr=lr, betas=(0.0, 0.99), eps=1e-8)
    # lazy R1 -> gamma scaled by interval to keep expected strength
    criterion = UNITPathSSLLoss(lambda1, lambda2, lambda3, gamma_g * r1_interval, gamma_s * r1_interval)
    aug = ada if ada is not None else (lambda t: t)
    d_params = list(disc_g.parameters()) + list(disc_s.parameters())

    iter_cnt = 0
    data_iter = iter(loader)
    print(f"\n===== Pretraining Stage {stage_num}: {max_iters} iters @ LR={lr} =====")

    while iter_cnt < max_iters:
        try:
            real_x, real_y = next(data_iter)
        except StopIteration:
            data_iter = iter(loader)
            real_x, real_y = next(data_iter)
        real_x, real_y = real_x.to(device), real_y.to(device)   # x: mask [0,1] | y: tissue [-1,1]
        do_r1 = (iter_cnt % r1_interval == 0)

        # 1. Discriminator step  (Eq.1 D_G with ADA + R1 ; Eq.2 D_S with R1 ; D_S fake = S(y))
        with torch.no_grad():
            fake_y = gen_g(real_x)                                # G(x)
            fake_x = torch.sigmoid(gen_s((real_y + 1.0) / 2.0))   # S(y)
        real_y_d = real_y.detach().requires_grad_(do_r1)
        real_x_d = real_x.detach().requires_grad_(do_r1)
        d_g_real, d_g_fake = disc_g(aug(real_y_d)), disc_g(aug(fake_y))
        d_g_loss, d_g_log = criterion.d_g_loss(d_g_real, d_g_fake, real_y_d if do_r1 else None)
        if ada is not None:
            ada.update(d_g_real.detach())
        d_s_real, d_s_fake = disc_s(real_x_d), disc_s(fake_x)
        d_s_loss, d_s_log = criterion.d_s_loss(d_s_real, d_s_fake, real_x_d if do_r1 else None)
        loss_d = (d_g_loss + d_s_loss) / accum_steps
        if iter_cnt % accum_steps == 0:
            opt_d.zero_grad(set_to_none=True)
        loss_d.backward()

        # 2. Generator step (G and S jointly) : adv_G + l1*adv_S + l2*(cyc_x + cyc_y)
        for p in d_params: p.requires_grad_(False)
        fake_y = gen_g(real_x)                                    # G(x)
        fake_x = torch.sigmoid(gen_s((real_y + 1.0) / 2.0))       # S(y)
        x_rec = torch.sigmoid(gen_s((fake_y + 1.0) / 2.0))        # S(G(x)) ~ x  (forward cycle)
        y_rec = gen_g(fake_x)                                     # G(S(y)) ~ y  (backward cycle)
        loss_g_total, g_log = criterion(disc_g(aug(fake_y)), disc_s(fake_x), x_rec, real_x, y_rec, real_y)
        loss_g_total = loss_g_total / accum_steps
        if iter_cnt % accum_steps == 0:
            opt_g.zero_grad(set_to_none=True)
        loss_g_total.backward()
        for p in d_params: p.requires_grad_(True)

        if (iter_cnt + 1) % accum_steps == 0:
            torch.nn.utils.clip_grad_norm_(list(gen_g.parameters()) + list(gen_s.parameters()), 1.0)
            torch.nn.utils.clip_grad_norm_(d_params, 1.0)
            opt_d.step()
            opt_g.step()

        iter_cnt += 1
        if iter_cnt % 500 == 0 or iter_cnt == max_iters:
            print(f"Stage {stage_num} | Iter [{iter_cnt:05d}/{max_iters:05d}] | "
                  f"G {loss_g_total.item() * accum_steps:.3f} (adv_g {g_log['loss_adv_g']:.3f} adv_s {g_log['loss_adv_s']:.3f} "
                  f"cyc_x {g_log['loss_cyc_x']:.3f} cyc_y {g_log['loss_cyc_y']:.3f}) | "
                  f"D_G {d_g_log['d_g_adv']:.3f} D_S {d_s_log['d_s_adv']:.3f} | ADA p={ada.p if ada is not None else 0:.2f}")

    ckpt_path = os.path.join(save_dir, f"{backbone}_stage{stage_num}_pretrained.pth")
    torch.save(gen_s.state_dict(), ckpt_path)
    print(f"--> Saved Stage {stage_num} checkpoint: {ckpt_path}")
    return ckpt_path

In [ ]:
# ============ 11. Run pretraining (skipped unless RUN_PRETRAINING) ============
if cfg.RUN_PRETRAINING:
    os.makedirs(cfg.pretrain_save_dir, exist_ok=True)
    pre_ds = LargeUnpairedDataset(cfg.pseudo_mask_dir, cfg.tissue_dir, cfg.pretrain_size)
    pre_loader = DataLoader(pre_ds, batch_size=cfg.pretrain_batch_size, shuffle=True,
                            num_workers=cfg.num_workers, pin_memory=True)

    gen_s = SegModel(backbone_name=cfg.backbone, pretrained=True).to(device)
    gen_g = CoModulatedGenerator(in_channels=1, out_channels=3).to(device)
    disc_g = PatchDiscriminator(in_channels=3).to(device)
    disc_s = PatchDiscriminator(in_channels=1).to(device)
    ada = ADA(target=cfg.ada_target).to(device) if cfg.use_ada else None
    pre_kw = dict(lambda1=cfg.pre_lambda1, lambda2=cfg.pre_lambda2, lambda3=cfg.pre_lambda3,
                  gamma_g=cfg.pre_gamma_g, gamma_s=cfg.pre_gamma_s, r1_interval=cfg.pre_r1_interval, ada=ada)

    run_stage(1, cfg.stage1_iters, 4.0e-4, gen_g, gen_s, disc_g, disc_s, pre_loader,
              device, cfg.pretrain_accum_steps, cfg.pretrain_save_dir, cfg.backbone, **pre_kw)

    # Stage 2: re-initialize S and D_S, keep G
    gen_s = SegModel(backbone_name=cfg.backbone, pretrained=True).to(device)
    disc_s = PatchDiscriminator(in_channels=1).to(device)
    stage2_ckpt = run_stage(2, cfg.stage2_iters, 1.0e-4, gen_g, gen_s, disc_g, disc_s, pre_loader,
                            device, cfg.pretrain_accum_steps, cfg.pretrain_save_dir, cfg.backbone, **pre_kw)

    cfg.pretrained_weights = stage2_ckpt   # feed into fine-tuning
    del gen_g, gen_s, disc_g, disc_s, pre_loader, pre_ds
    torch.cuda.empty_cache()
else:
    print("Pretraining skipped (RUN_PRETRAINING=False).")
    print("Fine-tuning will use",
          f"UNITPathSSL checkpoint: {cfg.pretrained_weights}" if cfg.pretrained_weights
          else "ImageNet initialization.")

## Stage C — MoNuSeg Dataset & Fine-tuning (`dataset.py`, `train.py`)
Includes all recent fixes:
- **Mask lookup** tries `{base}_bin_mask.png` → `{base}.png` → `{base} .png` (trailing-space names in the Kaggle dataset), with `os.path.exists` checks so OpenCV prints no warnings.
- **Rotate-and-crop augmentation:** the random scale is clamped to `cosθ+sinθ` so the rotated image always covers the full canvas — no mirror seams or black border wedges, and image/mask stay perfectly aligned.

In [ ]:
# ============ 12. MoNuSeg dataset (dataset.py, with recent fixes) ============
def list_image_files(img_dir, extensions=None):
    if extensions is None:
        extensions = (".png", ".tif", ".tiff", ".jpg", ".jpeg")
    return sorted([f for f in os.listdir(img_dir) if f.lower().endswith(extensions)])


def split_image_list(image_files, test_size=0.1, val_size=0.1, random_state=None):
    if val_size < 0 or test_size < 0 or val_size + test_size >= 1.0:
        raise ValueError("val_size and test_size must be non-negative and sum to less than 1.0")
    train_val, test_files = train_test_split(image_files, test_size=test_size,
                                             random_state=random_state, shuffle=True)
    if val_size == 0.0:
        return train_val, [], test_files
    val_fraction = val_size / (1.0 - test_size)
    train_files, val_files = train_test_split(train_val, test_size=val_fraction,
                                              random_state=random_state, shuffle=True)
    return train_files, val_files, test_files


class MoNuSegDataset(Dataset):
    def __init__(self, img_dir, mask_dir, target_size=(512, 512), augment=False,
                 augment_factor=1, augment_prob=0.5, rotation_range=(-15, 15),
                 scale_range=(0.9, 1.1), image_list=None):
        self.img_dir, self.mask_dir = img_dir, mask_dir
        self.target_size = tuple(target_size)
        self.augment = augment
        self.augment_factor = augment_factor
        self.augment_probability = augment_prob
        self.rotation_range = rotation_range
        self.scale_range = scale_range

        valid_extensions = (".png", ".tif", ".tiff", ".jpg", ".jpeg")
        if image_list is None:
            self.images = list_image_files(img_dir, extensions=valid_extensions)
        else:
            self.images = [f for f in image_list if f.lower().endswith(valid_extensions)]

        self.samples = []
        for img_name in self.images:
            self.samples.append({"img_name": img_name, "augment": False})
            if self.augment:
                for _ in range(self.augment_factor):
                    self.samples.append({"img_name": img_name, "augment": True})
        if self.augment:
            np.random.shuffle(self.samples)

    def __len__(self):
        return len(self.samples)

    def _augment_pair(self, img, mask):
        angle = np.random.uniform(self.rotation_range[0], self.rotation_range[1])
        scale = np.random.uniform(self.scale_range[0], self.scale_range[1])
        h, w = img.shape[:2]
        center = (w / 2.0, h / 2.0)

        # Rotate-and-crop: zoom so the rotated image always covers the canvas
        rad = np.deg2rad(abs(angle))
        cover_scale = np.cos(rad) + np.sin(rad)
        scale = max(scale, cover_scale)

        matrix = cv2.getRotationMatrix2D(center, angle, scale)
        img = cv2.warpAffine(img, matrix, (w, h), flags=cv2.INTER_LINEAR)
        mask = cv2.warpAffine(mask, matrix, (w, h), flags=cv2.INTER_NEAREST)
        return img, mask

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img_name = sample["img_name"]
        should_augment = sample["augment"]

        img_path = os.path.join(self.img_dir, img_name)
        img = cv2.imread(img_path)
        if img is None:
            raise FileNotFoundError(f"Could not read image: {img_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        base_name = os.path.splitext(img_name)[0]
        mask_candidates = (f"{base_name}_bin_mask.png", f"{base_name}.png", f"{base_name} .png")
        mask = None
        for mask_name in mask_candidates:
            mask_path = os.path.join(self.mask_dir, mask_name)
            if os.path.exists(mask_path):
                mask = cv2.imread(mask_path, 0)
                if mask is not None:
                    break
        if mask is None:
            raise FileNotFoundError(f"Could not find mask for {img_name} at {self.mask_dir}")

        img = cv2.resize(img, self.target_size, interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, self.target_size, interpolation=cv2.INTER_NEAREST)

        if should_augment and (np.random.rand() < self.augment_probability):
            img, mask = self._augment_pair(img, mask)

        img = np.ascontiguousarray(img)
        mask = np.ascontiguousarray(mask)

        mask = (mask > 0).astype("float32")
        img_tensor = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        mask_tensor = torch.from_numpy(mask).unsqueeze(0)
        return img_tensor, mask_tensor

In [ ]:
# ============ 13. Augmentation demo: before vs. after ============
demo_ds = MoNuSegDataset(cfg.train_img_dir, cfg.train_mask_dir,
                         target_size=cfg.target_size, augment=False)

n_examples = 3
np.random.seed(7)  # remove for different transforms each run
fig, axes = plt.subplots(n_examples, 4, figsize=(16, 4 * n_examples))

for row in range(n_examples):
    img_name = demo_ds.images[row]
    img = cv2.imread(os.path.join(demo_ds.img_dir, img_name))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    base = os.path.splitext(img_name)[0]
    mask = None
    for cand in (f"{base}_bin_mask.png", f"{base}.png", f"{base} .png"):
        cand_path = os.path.join(demo_ds.mask_dir, cand)
        if os.path.exists(cand_path):
            mask = cv2.imread(cand_path, 0)
            if mask is not None:
                break
    img = cv2.resize(img, demo_ds.target_size, interpolation=cv2.INTER_LINEAR)
    mask = cv2.resize(mask, demo_ds.target_size, interpolation=cv2.INTER_NEAREST)

    aug_img, aug_mask = demo_ds._augment_pair(img.copy(), mask.copy())

    axes[row, 0].imshow(img);                   axes[row, 0].set_title(f"Original\n{img_name}", fontsize=9)
    axes[row, 1].imshow(mask, cmap="gray");     axes[row, 1].set_title("Original Mask")
    axes[row, 2].imshow(aug_img);               axes[row, 2].set_title("Augmented (rotate + crop)")
    axes[row, 3].imshow(aug_mask, cmap="gray"); axes[row, 3].set_title("Augmented Mask")
    for ax in axes[row]:
        ax.axis("off")

plt.suptitle(f"Rotation {cfg.rotation_range}°, Scale {cfg.scale_range} "
             f"(rotate-and-crop: scale clamped to cover canvas, no border fill)", y=1.0)
plt.tight_layout(); plt.show()

In [ ]:
# ============ 14. Fine-tuning helpers + loaders + model (train.py) ============
def dice_score(preds, targets, smooth=1e-6):
    preds = preds.astype(np.float32)
    targets = targets.astype(np.float32)
    intersection = np.sum(preds * targets)
    return (2.0 * intersection + smooth) / (np.sum(preds) + np.sum(targets) + smooth)


def set_bn_eval(m):
    if "BatchNorm" in m.__class__.__name__:
        m.eval()


def evaluate(model, loader, device):
    model.eval()
    running_loss = 0.0
    all_preds_bin, all_masks, dice_scores = [], [], []
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            running_loss += supervised_loss(preds, masks).item()
            probs = torch.sigmoid(preds)
            pred_bin = (probs > 0.5).float().cpu().numpy()
            mask_np = masks.cpu().numpy()
            dice_scores.append(dice_score(pred_bin, mask_np))
            all_preds_bin.append(pred_bin.astype(np.uint8).flatten())
            all_masks.append(mask_np.astype(np.uint8).flatten())
    all_masks = np.concatenate(all_masks)
    all_preds_bin = np.concatenate(all_preds_bin)
    return (running_loss / len(loader), np.mean(dice_scores),
            jaccard_score(all_masks, all_preds_bin, zero_division=0),
            accuracy_score(all_masks, all_preds_bin),
            precision_score(all_masks, all_preds_bin, zero_division=0),
            recall_score(all_masks, all_preds_bin, zero_division=0),
            f1_score(all_masks, all_preds_bin, zero_division=0))


all_files = list_image_files(cfg.train_img_dir)
train_files, val_files, _ = split_image_list(all_files, test_size=0.1, val_size=0.15, random_state=42)
print(f"Split: {len(train_files)} train / {len(val_files)} val")

train_ds = MoNuSegDataset(cfg.train_img_dir, cfg.train_mask_dir, target_size=cfg.target_size,
                          augment=cfg.augment, augment_factor=cfg.augment_factor,
                          augment_prob=cfg.augment_prob, rotation_range=cfg.rotation_range,
                          scale_range=cfg.scale_range, image_list=train_files)
val_ds = MoNuSegDataset(cfg.train_img_dir, cfg.train_mask_dir, target_size=cfg.target_size,
                        augment=False, image_list=val_files)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=cfg.num_workers, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                        num_workers=cfg.num_workers, pin_memory=True)

model = SegModel(backbone_name=cfg.backbone, pretrained=cfg.pretrained,
                 num_classes=cfg.num_classes).to(device)

if cfg.pretrained_weights and os.path.exists(cfg.pretrained_weights):
    print(f"Loading UNITPathSSL pretrained checkpoint: {cfg.pretrained_weights}")
    checkpoint = torch.load(cfg.pretrained_weights, map_location=device)
    state_dict = checkpoint["state_dict"] if "state_dict" in checkpoint else checkpoint
    model.load_state_dict(state_dict, strict=False)

optimizer = optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs, eta_min=1e-6)

In [ ]:
# ============ 15. Fine-tuning loop (train.py) ============
best_val_dice = 0.0
best_checkpoint = os.path.join(cfg.save_dir, f"{cfg.backbone}_best.pth")
last_checkpoint = os.path.join(cfg.save_dir, f"{cfg.backbone}_last.pth")
log_csv_path = os.path.join(cfg.save_dir, f"{cfg.backbone}_training_metrics.csv")

csv_headers = ["epoch", "train_loss", "val_loss", "val_dice", "val_iou", "val_accuracy",
               "val_precision", "val_recall", "val_f1", "lr", "epoch_time_sec"]
with open(log_csv_path, mode="w", newline="") as f:
    csv.writer(f).writerow(csv_headers)

print(f"Starting training: Backbone={cfg.backbone}, Epochs={cfg.epochs}, "
      f"BatchSize={cfg.batch_size}, LR={cfg.lr}")

epoch_durations = []
total_start_time = time.time()

for epoch in range(1, cfg.epochs + 1):
    epoch_start_time = time.time()
    model.train()
    if cfg.freeze_bn:
        model.apply(set_bn_eval)
    running_loss = 0.0
    optimizer.zero_grad()

    for i, (imgs, masks) in enumerate(train_loader):
        imgs, masks = imgs.to(device), masks.to(device)
        preds = model(imgs)
        loss = supervised_loss(preds, masks) / cfg.accumulation_steps
        loss.backward()
        if (i + 1) % cfg.accumulation_steps == 0 or (i + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
        running_loss += loss.item() * cfg.accumulation_steps

    current_lr = optimizer.param_groups[0]["lr"]
    scheduler.step()

    train_loss = running_loss / len(train_loader)
    val_loss, val_dice, val_iou, val_acc, val_prec, val_rec, val_f1 = evaluate(model, val_loader, device)

    epoch_elapsed = time.time() - epoch_start_time
    epoch_durations.append(epoch_elapsed)

    with open(log_csv_path, mode="a", newline="") as f:
        csv.writer(f).writerow([
            epoch, f"{train_loss:.6f}", f"{val_loss:.6f}", f"{val_dice:.6f}",
            f"{val_iou:.6f}", f"{val_acc:.6f}", f"{val_prec:.6f}", f"{val_rec:.6f}",
            f"{val_f1:.6f}", f"{current_lr:.8f}", f"{epoch_elapsed:.2f}"])

    print(f"Epoch [{epoch:03d}/{cfg.epochs:03d}] ({epoch_elapsed:.1f}s) | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Val Dice: {val_dice:.4f} | IoU: {val_iou:.4f} | Acc: {val_acc:.4f}")

    torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(), "val_dice": val_dice},
               last_checkpoint)

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save(model.state_dict(), best_checkpoint)
        print(f"--> Saved Best Model Checkpoint (Val Dice: {best_val_dice:.4f})")

total_elapsed = time.time() - total_start_time
print(f"\n==================== Training Summary ====================")
print(f"Total Training Time:    {total_elapsed/60:.2f} mins")
print(f"Average Time / Epoch:   {sum(epoch_durations)/len(epoch_durations):.2f}s")
print(f"Best Validation Dice:   {best_val_dice:.4f}")
print(f"Best Checkpoint:        {best_checkpoint}")

## Stage D — Held-out Test Evaluation (`test.py`)

In [ ]:
# ============ 16. Test set evaluation (test.py) ============
test_files = list_image_files(cfg.test_img_dir)
test_ds = MoNuSegDataset(cfg.test_img_dir, cfg.test_mask_dir, target_size=cfg.target_size,
                         augment=False, image_list=test_files)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False,
                         num_workers=cfg.num_workers)

test_model = SegModel(backbone_name=cfg.backbone, pretrained=False).to(device)
test_model.load_state_dict(torch.load(best_checkpoint, map_location=device))
test_model.eval()

all_preds, all_masks_t, all_probs, dice_list = [], [], [], []
running_loss = 0.0
print(f"Evaluating {len(test_files)} test images using: {best_checkpoint}")

with torch.no_grad():
    for imgs, masks in test_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        preds = test_model(imgs)
        running_loss += supervised_loss(preds, masks).item()
        probs = torch.sigmoid(preds)
        pred_bin = (probs > 0.5).float().cpu().numpy()
        mask_np = masks.cpu().numpy()
        dice_list.append(dice_score(pred_bin, mask_np))
        all_probs.append(probs.cpu().numpy().flatten())
        all_preds.append(pred_bin.astype(np.uint8).flatten())
        all_masks_t.append(mask_np.astype(np.uint8).flatten())

all_masks_t = np.concatenate(all_masks_t)
all_preds = np.concatenate(all_preds)
all_probs = np.concatenate(all_probs)

print("\n================== TEST SET EVALUATION RESULTS ==================")
print(f"Mean Dice Score  : {np.mean(dice_list):.4f}")
print(f"Mean IoU (Jaccard): {jaccard_score(all_masks_t, all_preds, zero_division=0):.4f}")
print(f"F1-Score         : {f1_score(all_masks_t, all_preds, zero_division=0):.4f}")
print(f"Precision        : {precision_score(all_masks_t, all_preds, zero_division=0):.4f}")
print(f"Recall           : {recall_score(all_masks_t, all_preds, zero_division=0):.4f}")
print(f"Accuracy         : {accuracy_score(all_masks_t, all_preds):.4f}")
print(f"AUC-ROC          : {roc_auc_score(all_masks_t, all_probs):.4f}")
print(f"Mean Loss        : {running_loss / len(test_loader):.4f}")
print("=================================================================")

## Stage E — Outputs
1. **Training curves** (`plot_curves.py`): 2×2 grid — losses, Dice/IoU, Acc/Prec/Recall, LR schedule.
2. **Qualitative panels** (`generate_output_mask.py`): `[Original | Ground Truth | Prediction]` triptychs saved to disk.
3. **Unlabeled inference** (`infer_unlabeled.py`, optional): predicted mask + green overlay for tiles without labels.
4. **Baseline comparison** (`compare_baseline.py`, optional): overlay val-Dice curves of scratch / ImageNet / UNITPathSSL runs.

In [ ]:
# ============ 17. Training curves (plot_curves.py) ============
df = pd.read_csv(log_csv_path)
epochs_x = df["epoch"]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(epochs_x, df["train_loss"], label="Train Loss", color="#d62728", linewidth=1.8)
axes[0, 0].plot(epochs_x, df["val_loss"], label="Val Loss", color="#8c1111", linestyle="--", linewidth=1.8)
axes[0, 0].set_title("Training vs. Validation Loss"); axes[0, 0].set_xlabel("Epoch"); axes[0, 0].set_ylabel("Loss")
axes[0, 0].legend(loc="upper right"); axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(epochs_x, df["val_dice"], label="Val Dice Score", color="#1f77b4", linewidth=1.8)
axes[0, 1].plot(epochs_x, df["val_iou"], label="Val IoU (Jaccard)", color="#0e3d64", linestyle="--", linewidth=1.8)
axes[0, 1].set_title("Validation Dice & IoU Over Epochs"); axes[0, 1].set_xlabel("Epoch"); axes[0, 1].set_ylabel("Score")
axes[0, 1].legend(loc="lower right"); axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(epochs_x, df["val_accuracy"], label="Accuracy", color="#2ca02c", linewidth=1.8)
axes[1, 0].plot(epochs_x, df["val_precision"], label="Precision", color="#17becf", linestyle="--", linewidth=1.8)
axes[1, 0].plot(epochs_x, df["val_recall"], label="Recall", color="#bcbd22", linestyle=":", linewidth=1.8)
axes[1, 0].set_title("Classification & Segmentation Accuracy"); axes[1, 0].set_xlabel("Epoch"); axes[1, 0].set_ylabel("Score")
axes[1, 0].legend(loc="lower right"); axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(epochs_x, df["lr"], label="Learning Rate", color="#9467bd", linewidth=1.8)
axes[1, 1].set_title("Cosine Annealing Learning Rate Schedule"); axes[1, 1].set_xlabel("Epoch"); axes[1, 1].set_ylabel("LR")
axes[1, 1].set_yscale("log"); axes[1, 1].legend(loc="upper right"); axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
curves_path = os.path.join(cfg.save_dir, "training_curves.png")
plt.savefig(curves_path, dpi=300)
plt.show()
print(f"--> Saved training curve plot to: {curves_path}")

In [ ]:
# ============ 18. Qualitative triple-pane panels (generate_output_mask.py) ============
def find_mask_path(mask_dir, base_name):
    candidates = [os.path.join(mask_dir, f"{base_name}_bin_mask.png"),
                  os.path.join(mask_dir, f"{base_name}.png"),
                  os.path.join(mask_dir, f"{base_name} .png"),   # trailing-space names
                  os.path.join(mask_dir, f"{base_name}_mask.png"),
                  os.path.join(mask_dir, f"{base_name}_bin_mask.tif"),
                  os.path.join(mask_dir, f"{base_name}.tif")]
    for path in candidates:
        if os.path.exists(path):
            return path
    return None


def process_set(model, img_dir, mask_dir, img_list, source_label, device, output_folder,
                target_size=(512, 512), display_size=(512, 512), threshold=0.5,
                title_tag="UNITPath Prediction"):
    """Saves [Original | Ground Truth | Prediction] panels to disk."""
    valid_count = 0
    for img_name in img_list:
        original_bgr = cv2.imread(os.path.join(img_dir, img_name))
        if original_bgr is None:
            continue
        base_name = os.path.splitext(img_name)[0]
        h, w = original_bgr.shape[:2]

        img_input = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2RGB)
        img_input = cv2.resize(img_input, tuple(target_size), interpolation=cv2.INTER_LINEAR)
        img_tensor = (torch.from_numpy(img_input).permute(2, 0, 1).float() / 255.0).unsqueeze(0).to(device)

        mask_path = find_mask_path(mask_dir, base_name)
        if mask_path is not None:
            gt_mask = (cv2.imread(mask_path, 0) > 0).astype(np.uint8) * 255
        else:
            gt_mask = np.zeros((h, w), dtype=np.uint8)

        with torch.no_grad():
            prob = torch.sigmoid(model(img_tensor)).squeeze().cpu().numpy()
        pred_mask = (prob > threshold).astype(np.uint8) * 255
        pred_mask = cv2.resize(pred_mask, (w, h), interpolation=cv2.INTER_NEAREST)

        disp_w, disp_h = display_size
        pane1 = cv2.resize(original_bgr, (disp_w, disp_h))
        pane2 = cv2.resize(cv2.cvtColor(gt_mask, cv2.COLOR_GRAY2BGR), (disp_w, disp_h))
        pane3 = cv2.resize(cv2.cvtColor(pred_mask, cv2.COLOR_GRAY2BGR), (disp_w, disp_h))

        gap = np.zeros((disp_h, 10, 3), dtype=np.uint8)
        combined = np.hstack((pane1, gap, pane2, gap, pane3))

        font, color = cv2.FONT_HERSHEY_SIMPLEX, (0, 255, 0)
        cv2.putText(combined, f"Source: {source_label}", (10, disp_h - 20), font, 0.7, (255, 255, 255), 2)
        cv2.putText(combined, "Original", (10, 35), font, 0.9, color, 2)
        cv2.putText(combined, "Ground Truth", (disp_w + 20, 35), font, 0.9, color, 2)
        cv2.putText(combined, title_tag, (2 * disp_w + 30, 35), font, 0.9, color, 2)

        save_path = os.path.join(output_folder, f"{source_label}_{base_name}.png")
        cv2.imwrite(save_path, combined)
        valid_count += 1
    return valid_count


n = process_set(test_model, cfg.test_img_dir, cfg.test_mask_dir, test_files, "Test",
                device, cfg.panels_dir, target_size=cfg.target_size,
                threshold=cfg.threshold, title_tag=cfg.title_tag)
print(f"Saved {n} panels to {cfg.panels_dir}")

# Display the first 3 panels inline
panel_files = sorted(glob.glob(os.path.join(cfg.panels_dir, "*.png")))[:3]
for pf in panel_files:
    plt.figure(figsize=(16, 6))
    plt.imshow(cv2.cvtColor(cv2.imread(pf), cv2.COLOR_BGR2RGB))
    plt.title(os.path.basename(pf)); plt.axis("off"); plt.show()

In [ ]:
# ============ 19. Inference on unlabeled tiles (infer_unlabeled.py) — optional ============
if cfg.RUN_UNLABELED_INFERENCE:
    os.makedirs(cfg.unlabeled_out_dir, exist_ok=True)
    valid_exts = (".png", ".tif", ".tiff", ".jpg", ".jpeg")
    tissue_files = sorted([f for f in glob.glob(os.path.join(cfg.tissue_dir, "**", "*.*"), recursive=True)
                           if f.lower().endswith(valid_exts)])
    print(f"Found {len(tissue_files)} unlabeled images. Processing first 30...")

    with torch.no_grad():
        for img_path in tissue_files[:30]:
            img = cv2.imread(img_path)
            if img is None:
                continue
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img_resized = cv2.resize(img_rgb, cfg.target_size, interpolation=cv2.INTER_LINEAR)
            img_tensor = (torch.from_numpy(img_resized).permute(2, 0, 1).float() / 255.0).unsqueeze(0).to(device)

            probs = torch.sigmoid(test_model(img_tensor)).squeeze().cpu().numpy()
            pred_bin = (probs > cfg.threshold).astype(np.uint8) * 255

            base_name = os.path.splitext(os.path.basename(img_path))[0]
            cv2.imwrite(os.path.join(cfg.unlabeled_out_dir, f"{base_name}_pred_mask.png"), pred_bin)

            overlay = img_resized.copy()
            overlay[pred_bin > 0] = [0, 255, 0]
            blended = cv2.addWeighted(img_resized, 0.6, overlay, 0.4, 0)
            comparison = np.hstack([cv2.cvtColor(img_resized, cv2.COLOR_RGB2BGR),
                                    cv2.cvtColor(pred_bin, cv2.COLOR_GRAY2BGR),
                                    cv2.cvtColor(blended, cv2.COLOR_RGB2BGR)])
            cv2.imwrite(os.path.join(cfg.unlabeled_out_dir, f"{base_name}_comparison.png"), comparison)
    print(f"Generated masks saved in: {cfg.unlabeled_out_dir}")
else:
    print("Unlabeled inference skipped (RUN_UNLABELED_INFERENCE=False).")

In [ ]:
# ============ 20. Baseline comparison (compare_baseline.py) — optional ============
# Overlays val-Dice convergence of multiple runs. Each run writes its own metrics CSV;
# re-run Stage C with different init (pretrained=False / ImageNet / UNITPathSSL ckpt)
# and different save_dir, then list the CSVs here.
baseline_runs = {
    # "Scratch (Random)":    "/kaggle/working/checkpoints/ablation_scratch/resnet50_training_metrics.csv",
    # "ImageNet Pretrained": "/kaggle/working/checkpoints/ablation_imagenet/resnet50_training_metrics.csv",
    "Current Run": log_csv_path,
}

plt.figure(figsize=(10, 6))
styles = [("gray", ":"), ("blue", "--"), ("green", "-")]
for (label, csv_file), (color, ls) in zip(baseline_runs.items(), styles):
    if os.path.exists(csv_file):
        d = pd.read_csv(csv_file)
        plt.plot(d["epoch"], d["val_dice"], label=label, color=color, linestyle=ls,
                 linewidth=2.5 if ls == "-" else 1.8)
plt.title("Validation Dice Score Convergence", fontsize=14)
plt.xlabel("Epoch"); plt.ylabel("Validation Dice")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig("/kaggle/working/checkpoints/pretrain_advancement_comparison.png", dpi=300)
plt.show()

## Outputs & Notes
- **Checkpoints + metrics CSV:** `/kaggle/working/checkpoints/final/`
- **Qualitative panels:** `/kaggle/working/output/test_panels/`
- All downloadable from the notebook **Output** tab after saving a version.

**Not included:** `RCNN.py` (auxiliary Mask R-CNN used as the Instance-Segmentation-Guided network — a separate experiment, torchvision-only so it can be added if needed), `grid_search.py` (KFold hyperparameter sweep — too long for one Kaggle session; the thesis hyperparameters in `Config` are its result).